In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# For the FS phase we select the standardized data, because it trasforms the
#   data in order to have 0 average and standard deviation equalt to 1. Hence,
#   standardization makes the different features comparable
import pandas as pd
import os

project_dir = '/content/drive/MyDrive/DSH - Lab'
lab_dir = os.path.join(project_dir, 'Lab7-8')

df = pd.read_csv(os.path.join(project_dir, 'Lab2', 'features_CS_binWidth10_standardized.csv'))

# Selecting robust features

In [4]:
# First, we exclude from the feature list those that are not robust, meaning
#   that they have an ICC < 0.8

icc_df = pd.read_csv(os.path.join(project_dir, 'Lab2', 'icc_feature_robustness.csv'))
robust_features = icc_df[icc_df['ICC(3,1)'] > 0.8]['feature'].tolist()

# Assuming 'ID' and 'DEFINITIVE DIAGNOSIS ' are columns to keep regardless of robustness
# Correcting the column name based on previous output
columns_to_keep = ['ID'] + robust_features + ['DEFINITIVE DIAGNOSIS ']
diagnosis_column = 'DEFINITIVE DIAGNOSIS '


df_robust = df[columns_to_keep].copy()

display(df_robust.head())

,ID,original_glcm_Contrast,original_glcm_DifferenceAverage,original_glcm_DifferenceVariance,original_glcm_InverseVariance,original_glcm_Id,original_glcm_Idm,original_gldm_DependenceVariance,original_glcm_DifferenceEntropy,original_ngtdm_Contrast,...,original_gldm_GrayLevelNonUniformity,original_glrlm_LongRunLowGrayLevelEmphasis,original_shape_MeshVolume,original_shape_VoxelVolume,original_shape_SurfaceArea,original_shape_Flatness,original_glszm_LowGrayLevelZoneEmphasis,original_shape_Maximum3DDiameter,original_shape_Maximum2DDiameterRow,DEFINITIVE DIAGNOSIS
0,1,-0.177452,-0.068193,-0.167363,-0.109518,-0.100363,-0.132810,0.030398,0.069502,-0.134018,...,-0.329643,-0.547511,-0.301058,-0.302143,-0.430263,0.661434,-0.321583,-0.380024,-0.157748,0
1,2,0.813641,0.914621,0.833037,-0.973839,-0.957156,-0.952059,-0.715613,0.988990,0.942123,...,0.084025,-0.154344,0.246013,0.247253,0.627339,-0.223457,-0.162592,1.086458,0.679360,0
2,3,0.301183,0.439140,0.318833,-0.614692,-0.586042,-0.608266,-0.699998,0.571961,0.270857,...,-0.313766,-0.810726,-0.267815,-0.265321,-0.068832,-0.422695,-0.852004,0.254550,0.220122,0
3,4,-0.081094,0.091005,-0.119505,-0.403133,-0.335300,-0.385686,-0.396772,0.219490,-0.215864,...,-0.467020,-0.246978,-0.376591,-0.376725,-0.523031,-0.420156,-0.157891,-0.674099,-0.638371,0
4,5,-0.362271,-0.264973,-0.377931,0.077420,0.084751,0.046559,0.062274,-0.152759,-0.423081,...,0.197947,-0.069448,0.197605,0.198983,0.591756,-0.007702,-0.140698,0.927745,1.036846,0


# Filter/Ranking algorithm - mRMR

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mutual_info_score
from sklearn.feature_selection import mutual_info_classif, mutual_info_regression

def mrmr_feature_selection(df, target_column):
    """
    Implements the mRMR (Minimum Redundancy Maximum Relevance) feature selection algorithm.
    It ranks all features with non-zero relevance to the target.

    Args:
        df (pd.DataFrame): The input DataFrame containing features and the target variable.
        target_column (str): The name of the target variable column.

    Returns:
        tuple: A tuple containing two lists: the ranked list of selected feature names,
               and a list of excluded feature names.
    """
    # Separate features (X) and target (y)
    y = df[target_column]
    X = df.drop(columns=[target_column, 'ID'], errors='ignore')

    # Get the list of all feature names
    feature_names = X.columns.tolist()

    # --- Step 1: Select the feature with the largest relevance ---
    # Calculate the mutual information between each feature and the target
    relevance_scores = mutual_info_classif(X, y, discrete_features=False)

    # Create a Series for easier lookup
    relevance = pd.Series(relevance_scores, index=feature_names)

    # Filter for features with non-zero relevance, as per the algorithm's stopping condition
    relevant_features = relevance[relevance > 0].index.tolist()

    if not relevant_features:
        # If no relevant features, return empty lists for both selected and excluded
        return [], feature_names

    # Find the feature with the maximum relevance among the relevant ones
    first_feature = relevance.loc[relevant_features].idxmax()

    # Initialize the set of selected features and the set of remaining features
    selected_features = [first_feature]
    remaining_features = [f for f in relevant_features if f != first_feature]

    # --- Steps 2, 3, 4: Iteratively add features that maximize the MIQ ---
    while remaining_features:
        miq_scores = {}

        # Calculate MIQ for each remaining feature
        for feature_x in remaining_features:
            # Maximum Relevance part (already calculated)
            relevance_term = relevance[feature_x]

            # Minimum Redundancy part
            redundancy_term = 0
            for feature_z in selected_features:
                # We calculate MI between two continuous features using mutual_info_regression
                # Input X needs to be 2D: (n_samples, 1)
                mi = mutual_info_regression(
                    X[[feature_x]],
                    X[feature_z],
                    discrete_features=False,
                    random_state=42
                )
                redundancy_term += mi[0]

            # Average the redundancy
            redundancy_term /= len(selected_features)

            # Calculate Mutual Information Quotient (MIQ)
            # Handle division by zero if redundancy is 0
            if redundancy_term > 0:
                miq = relevance_term / redundancy_term
            else:
                # If redundancy is 0, we can assign a very large value to prioritize it
                miq = np.inf

            miq_scores[feature_x] = miq

        # Select the feature with the maximum MIQ
        best_feature = max(miq_scores, key=miq_scores.get)

        # Add the best feature to the selected list and remove it from the remaining list
        selected_features.append(best_feature)
        remaining_features.remove(best_feature)

    # Determine excluded features
    all_features = X.columns.tolist()
    excluded_features = [f for f in all_features if f not in selected_features]


    return selected_features, excluded_features


if __name__ == '__main__':
    # Define the target column
    target = 'DEFINITIVE DIAGNOSIS '

    # Run the mRMR algorithm to get a ranked list of all relevant features and the excluded ones
    ranked_features, excluded_features = mrmr_feature_selection(
        df=df_robust,
        target_column=target
    )

    print(f"Features selected and ranked by mRMR: {len(ranked_features)}")
    print(ranked_features)

    print(f"\nFeatures excluded by mRMR: {len(excluded_features)}")
    print(excluded_features)

    # Save the ranked features to a CSV file
    ranked_features_df = pd.DataFrame({'ranked_features': ranked_features})
    ranked_features_df.to_csv(os.path.join(lab_dir, 'mRMR-features.csv'), index=False)

    print("\nRanked features saved to '/content/drive/MyDrive/DSH - Lab/Lab7-8/mRMR-features.csv'")

    # Merge with the original ICC dataframe to get the ICC values
    # We use 'how=left' to keep the order of the ranked features
    mrmr_final_df = ranked_features_df.merge(icc_df[['feature', 'ICC(3,1)']], left_on='ranked_features', right_on='feature', how='left')
    mrmr_final_df.drop(columns=['feature'], inplace=True)

    # Save to CSV
    output_path_mrmr = os.path.join(lab_dir, 'mRMR-features_with_ICC.csv')
    mrmr_final_df.to_csv(output_path_mrmr, index=False)

    print("\nRanked features with ICC saved to '/content/drive/MyDrive/DSH - Lab/Lab7-8/mRMR-features_with_ICC.csv'")

Features selected and ranked by mRMR: 57
['original_shape_SurfaceVolumeRatio', 'original_glcm_Contrast', 'original_shape_MajorAxisLength', 'original_shape_Flatness', 'original_glcm_Imc2', 'original_shape_SurfaceArea', 'original_gldm_LargeDependenceHighGrayLevelEmphasis', 'original_firstorder_TotalEnergy', 'original_glszm_ZonePercentage', 'original_shape_Maximum2DDiameterSlice', 'original_glszm_GrayLevelNonUniformityNormalized', 'original_glcm_Imc1', 'original_shape_MinorAxisLength', 'original_firstorder_Mean', 'original_glszm_SizeZoneNonUniformity', 'original_ngtdm_Contrast', 'original_shape_MeshVolume', 'original_shape_Maximum2DDiameterRow', 'original_glcm_Correlation', 'original_shape_LeastAxisLength', 'original_firstorder_Energy', 'original_shape_Maximum2DDiameterColumn', 'original_glcm_MCC', 'original_glcm_SumEntropy', 'original_shape_Maximum3DDiameter', 'original_shape_VoxelVolume', 'original_glrlm_RunLengthNonUniformity', 'original_firstorder_90Percentile', 'original_gldm_SmallDe

## Results from mRMR

Features selected and ranked by mRMR: 57
['original_shape_SurfaceVolumeRatio', 'original_glcm_Contrast', 'original_shape_MajorAxisLength', 'original_shape_Flatness', 'original_firstorder_Median', 'original_glcm_Imc2', 'original_shape_SurfaceArea', 'original_gldm_LargeDependenceHighGrayLevelEmphasis', 'original_firstorder_TotalEnergy', 'original_glszm_ZonePercentage', 'original_shape_Maximum2DDiameterSlice', 'original_glszm_GrayLevelNonUniformityNormalized', 'original_glcm_Imc1', 'original_shape_MinorAxisLength', 'original_glszm_SizeZoneNonUniformity', 'original_firstorder_90Percentile', 'original_shape_MeshVolume', 'original_ngtdm_Contrast', 'original_shape_Maximum2DDiameterRow', 'original_glcm_Correlation', 'original_firstorder_Energy', 'original_shape_Maximum2DDiameterColumn', 'original_shape_LeastAxisLength', 'original_glcm_MCC', 'original_shape_Maximum3DDiameter', 'original_glcm_SumEntropy', 'original_shape_VoxelVolume', 'original_glrlm_RunLengthNonUniformity', 'original_gldm_SmallDependenceEmphasis', 'original_ngtdm_Strength', 'original_glszm_GrayLevelNonUniformity', 'original_ngtdm_Coarseness', 'original_gldm_GrayLevelNonUniformity', 'original_firstorder_Uniformity', 'original_gldm_DependenceNonUniformity', 'original_firstorder_Mean', 'original_ngtdm_Complexity', 'original_glrlm_GrayLevelNonUniformity', 'original_glcm_Id', 'original_ngtdm_Busyness', 'original_gldm_DependenceEntropy', 'original_glcm_DifferenceAverage', 'original_firstorder_Entropy', 'original_glszm_SizeZoneNonUniformityNormalized', 'original_glcm_Idm', 'original_glcm_Idn', 'original_gldm_GrayLevelVariance', 'original_glcm_SumSquares', 'original_glrlm_RunPercentage', 'original_glrlm_GrayLevelNonUniformityNormalized', 'original_glcm_DifferenceEntropy', 'original_glcm_ClusterTendency', 'original_glszm_GrayLevelVariance', 'original_firstorder_InterquartileRange', 'original_firstorder_Variance', 'original_glrlm_RunEntropy', 'original_glrlm_RunLengthNonUniformityNormalized']

Features excluded by mRMR: 29
['original_glcm_DifferenceVariance', 'original_glcm_InverseVariance', 'original_gldm_DependenceVariance', 'original_glrlm_RunVariance', 'original_gldm_LargeDependenceEmphasis', 'original_glrlm_LongRunEmphasis', 'original_gldm_DependenceNonUniformityNormalized', 'original_glrlm_ShortRunEmphasis', 'original_glcm_Idmn', 'original_glcm_JointEntropy', 'original_firstorder_RobustMeanAbsoluteDeviation', 'original_glrlm_GrayLevelVariance', 'original_firstorder_MeanAbsoluteDeviation', 'original_glcm_ClusterProminence', 'original_glcm_JointEnergy', 'original_glcm_MaximumProbability', 'original_gldm_SmallDependenceHighGrayLevelEmphasis', 'original_firstorder_RootMeanSquared', 'original_gldm_SmallDependenceLowGrayLevelEmphasis', 'original_firstorder_10Percentile', 'original_glszm_SmallAreaEmphasis', 'original_glrlm_ShortRunLowGrayLevelEmphasis', 'original_gldm_LowGrayLevelEmphasis', 'original_glrlm_LowGrayLevelRunEmphasis', 'original_glszm_ZoneEntropy', 'original_shape_Sphericity', 'original_glszm_SmallAreaLowGrayLevelEmphasis', 'original_glrlm_LongRunLowGrayLevelEmphasis', 'original_glszm_LowGrayLevelZoneEmphasis']

Ranked features saved to '/content/drive/MyDrive/DSH - Lab/Lab7-8/mRMR-features.csv'

# Filter method - QRA

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import KBinsDiscretizer
from collections import defaultdict
import os

# --- 1. Funzioni Helper per l'Algoritmo QuickReduct ---

def get_equivalence_classes(data):
    """
    Calcola le classi di equivalenza (gruppi di indici di righe identiche).
    Input: un array numpy 2D (campioni x feature).
    Output: un dizionario dove le chiavi sono tuple (le righe)
            e i valori sono set di indici (le classi).
    """
    classes = defaultdict(set)
    for i in range(data.shape[0]):
        # Convertiamo la riga in una tupla per renderla "hashable"
        # e usarla come chiave del dizionario
        row_tuple = tuple(data[i])
        classes[row_tuple].add(i)
    return classes

def get_dependency_degree(X_subset_discrete, y_classes, n_samples):
    """
    Calcola il grado di dipendenza (gamma) tra un sottoinsieme
    di feature X e le classi target y.

    gamma = |POS(X, y)| / |U|

    dove POS è la "Positive Region": l'unione delle classi di equivalenza
    di X che sono interamente contenute in *una* classe di y.
    |U| è il numero totale di campioni.
    """

    # Se il subset è vuoto, gestiamo il caso base
    if X_subset_discrete.shape[1] == 0:
        # Crea un array "vuoto" con il giusto numero di campioni
        X_subset_discrete = np.empty((n_samples, 0))

    # 1. Trova le classi di equivalenza per il subset di feature X
    x_classes = get_equivalence_classes(X_subset_discrete)

    positive_region_size = 0

    # 2. Per ogni classe di X, controlla se è un sottoinsieme di una classe di y
    for x_class in x_classes.values():
        for y_class in y_classes.values():
            if x_class.issubset(y_class):
                # Se lo è, tutti i campioni in questa classe x
                # appartengono alla "positive region"
                positive_region_size += len(x_class)
                break # Passa alla prossima classe x

    # 3. Calcola il grado di dipendenza
    gamma = positive_region_size / n_samples
    return gamma

# --- 2. Algoritmo QuickReduct Principale ---

def quick_reduct(X_discrete, y):
    """
    Implementa l'algoritmo QuickReduct.
    Input:
        X_discrete: Matrice (numpy) di feature GIA' DISCRETIZZATE
        y: Array (numpy) 1D della target
    Output:
        Un set di INDICI delle colonne che formano il "ridotto".
    """

    n_samples, n_features = X_discrete.shape

    # Pre-calcola le classi della target (non cambiano)
    # Rimodelliamo y in 2D per compatibilità con la funzione helper
    y_reshaped = y.values.reshape(-1, 1)
    y_classes = get_equivalence_classes(y_reshaped)

    # Indici di tutte le feature disponibili
    all_feature_indices = set(range(n_features))

    # Inizia con un "ridotto" vuoto
    reduct = set()

    # Calcola la dipendenza massima possibile (usando tutte le feature)
    gamma_max = get_dependency_degree(X_discrete, y_classes, n_samples)

    # Calcola la dipendenza attuale (con un set vuoto)
    gamma_current = get_dependency_degree(np.empty((n_samples, 0)), y_classes, n_samples)

    print(f"Dipendenza massima (con tutte le feature): {gamma_max:.4f}")

    # Loop finché non raggiungiamo la dipendenza massima
    while gamma_current < gamma_max:
        best_feature_to_add = -1
        best_new_gamma = gamma_current

        # Feature ancora da testare
        potential_features = all_feature_indices - reduct

        if not potential_features:
            break # Usciamo se abbiamo testato tutto

        # Algoritmo Greedy: testa ogni feature non ancora nel ridotto
        for feature_idx in potential_features:

            # Prova ad aggiungere la feature al ridotto attuale
            temp_reduct_indices = list(reduct | {feature_idx})
            X_temp_subset = X_discrete[:, temp_reduct_indices]

            # Calcola la dipendenza con la nuova feature
            temp_gamma = get_dependency_degree(X_temp_subset, y_classes, n_samples)

            # Se questa feature dà la dipendenza migliore finora, salvala
            if temp_gamma > best_new_gamma:
                best_new_gamma = temp_gamma
                best_feature_to_add = feature_idx

        # Se abbiamo trovato una feature che migliora la dipendenza...
        if best_feature_to_add != -1:
            reduct.add(best_feature_to_add)
            gamma_current = best_new_gamma
            # print(f"  Aggiunta feature {best_feature_to_add}, Nuova dipendenza: {gamma_current:.4f}")
        else:
            # Nessuna feature migliora la dipendenza,
            # siamo bloccati in un ottimo locale o gamma_max è irraggiungibile
            print("Nessuna feature migliora ulteriormente la dipendenza.")
            break

    return reduct


# --- FASE A: Preparazione e Discretizzazione Dati ---

# Ensure consistent column names by stripping whitespace for ALL columns
df_robust.columns = df_robust.columns.str.strip()

# Find the actual diagnosis column name (could be 'DEFINITIVE DIAGNOSIS' after stripping or 'DIAGNOSIS' if already renamed)
found_diagnosis_col_name = None
if 'DEFINITIVE DIAGNOSIS' in df_robust.columns:
    found_diagnosis_col_name = 'DEFINITIVE DIAGNOSIS'
elif 'DIAGNOSIS' in df_robust.columns:
    found_diagnosis_col_name = 'DIAGNOSIS'

if found_diagnosis_col_name is None:
    raise KeyError(f"Target column 'DEFINITIVE DIAGNOSIS' or 'DIAGNOSIS' not found in df_robust after stripping names. Current columns: {df_robust.columns.tolist()}")

# Standardize the target column name to 'DIAGNOSIS'
if found_diagnosis_col_name != 'DIAGNOSIS':
    df_robust.rename(columns={found_diagnosis_col_name: 'DIAGNOSIS'}, inplace=True)

target_col = 'DIAGNOSIS' # Set target_col to the standardized name
id_col = 'ID'

# Identifica le colonne delle feature
feature_cols = [col for col in df_robust.columns if col not in [target_col, id_col]]

X_continuous = df_robust[feature_cols]
y = df_robust[target_col]

# Inizializza il discretizzatore
# n_bins=4 : divide ogni feature in 4 intervalli
# strategy='uniform': gli intervalli hanno la stessa ampiezza
# encode='ordinal': assegna a ogni intervallo un numero (0, 1, 2, 3)
discretizer = KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='uniform', subsample=None)

# Applica la discretizzazione
# Usiamo .values per passare un array numpy, che è più robusto
X_discrete = discretizer.fit_transform(X_continuous.values).astype(int)

print(f"Feature originali: {feature_cols}")
print("\n--- Dati dopo la Discretizzazione (X_discrete) ---")
print(X_discrete)
print("-" * 30)

# Create DataFrame for discretized features
discretized_df_temp = pd.DataFrame(X_discrete, columns=feature_cols)

# Add ID and target column to the discretized features
discretized_df = df_robust[[id_col, target_col]].reset_index(drop=True)
discretized_df = pd.concat([discretized_df, discretized_df_temp], axis=1)

output_dir_qra = lab_dir
os.makedirs(output_dir_qra, exist_ok=True)
discretized_df.to_csv(os.path.join(output_dir_qra, 'QRA-discretized_features.csv'), index=False)
print(f"\nDiscretized features saved to '{os.path.join(output_dir_qra, 'QRA-discretized_features.csv')}'")


# --- FASE B: Esecuzione di QuickReduct ---

print("\nAvvio Algoritmo QuickReduct...")
selected_feature_indices = quick_reduct(X_discrete, y)

print("-" * 30)
print(f"Indici delle feature selezionate: {selected_feature_indices}")

# Mappa gli indici ai nomi originali delle colonne
selected_feature_names = [feature_cols[i] for i in selected_feature_indices]

print(f"\nNomi delle feature selezionate (Ridotto): {selected_feature_names}")

# Save the ranked features to a CSV file
selected_feature_names_df = pd.DataFrame({'selected_feature_names': selected_feature_names})
selected_feature_names_df.to_csv(os.path.join(output_dir_qra, 'QRA-features.csv'), index=False)

print(f"\nRanked features saved to '{os.path.join(output_dir_qra, 'QRA-features.csv')}'")

# Merge with the original ICC dataframe to get the ICC values
# We use 'how=left' to keep the order of the ranked features
qra_final_df = selected_feature_names_df.merge(icc_df[['feature', 'ICC(3,1)']], left_on='selected_feature_names', right_on='feature', how='left')
qra_final_df.drop(columns=['feature'], inplace=True)

# Save to CSV
output_path_qra_icc = os.path.join(output_dir_qra, 'QRA-features_with_ICC.csv')
qra_final_df.to_csv(output_path_qra_icc, index=False)

print(f"\nRanked features with ICC saved to '{output_path_qra_icc}'")

Feature originali: ['original_glcm_Contrast', 'original_glcm_DifferenceAverage', 'original_glcm_DifferenceVariance', 'original_glcm_InverseVariance', 'original_glcm_Id', 'original_glcm_Idm', 'original_gldm_DependenceVariance', 'original_glcm_DifferenceEntropy', 'original_ngtdm_Contrast', 'original_glrlm_RunVariance', 'original_gldm_LargeDependenceEmphasis', 'original_glrlm_LongRunEmphasis', 'original_gldm_DependenceNonUniformityNormalized', 'original_glcm_SumSquares', 'original_glrlm_RunPercentage', 'original_glrlm_ShortRunEmphasis', 'original_glcm_Idmn', 'original_glrlm_RunLengthNonUniformityNormalized', 'original_glcm_JointEntropy', 'original_firstorder_RobustMeanAbsoluteDeviation', 'original_firstorder_InterquartileRange', 'original_gldm_SmallDependenceEmphasis', 'original_gldm_GrayLevelVariance', 'original_firstorder_Variance', 'original_glszm_ZonePercentage', 'original_glrlm_GrayLevelVariance', 'original_firstorder_MeanAbsoluteDeviation', 'original_glcm_Idn', 'original_firstorder_

## QRA output

Dati dopo la Discretizzazione (X_discrete)
[[0 1 1 ... 0 0 1]
 [1 2 1 ... 1 2 1]
 [1 1 1 ... 0 1 1]
 ...
 [0 1 0 ... 0 1 1]
 [0 1 0 ... 1 1 1]
 [0 0 0 ... 0 1 1]]


Discretized features saved to '/content/drive/MyDrive/DSH - Lab/Lab7-8/QRA-discretized_features.csv'

Avvio Algoritmo QuickReduct...
Dipendenza massima (con tutte le feature): 1.0000


Indici delle feature selezionate: {0, 35, 67, 16, 82, 19, 84, 54}

Nomi delle feature selezionate (Ridotto): ['original_glcm_Contrast', 'original_glcm_MCC', 'original_shape_Sphericity', 'original_glcm_Idmn', 'original_shape_Flatness', 'original_firstorder_RobustMeanAbsoluteDeviation', 'original_shape_Maximum3DDiameter', 'original_shape_SurfaceVolumeRatio']

Ranked features saved to '/content/drive/MyDrive/DSH - Lab/Lab7-8/QRA-features.csv'

Ranked features with ICC saved to '/content/drive/MyDrive/DSH - Lab/Lab7-8/QRA-features_with_ICC.csv'

# Filter method - EBR

In [ ]:
# ============================================================
# 🔬 ENTROPY-BASED REDUCTION (EBR)
# Versione completa con discretizzazione automatica
# ============================================================

import pandas as pd
import numpy as np
from sklearn.preprocessing import KBinsDiscretizer
from collections import defaultdict
import os

# ============================================================
# 1️⃣ FUNZIONE: Calcolo dell'entropia condizionale H(Y|X)
# ============================================================

def calculate_conditional_entropy(df, feature_subset, target_col):
    """
    Calcola l'entropia condizionale H(Y|X), cioè l'incertezza del target Y
    conoscendo una o più feature X.
    """
    if not feature_subset:
        # Entropia base del target (H(Y))
        probs = df[target_col].value_counts(normalize=True)
        return -np.sum([p * np.log2(p) for p in probs if p > 0])

    total_entropy = 0
    total_samples = len(df)
    grouped = df.groupby(feature_subset)

    for _, group in grouped:
        group_size = len(group)
        prob_group = group_size / total_samples  # P(x_j)
        probs_in_group = group[target_col].value_counts(normalize=True)
        entropy_in_group = -np.sum([p * np.log2(p) for p in probs_in_group if p > 0])
        total_entropy += prob_group * entropy_in_group

    return total_entropy


# ============================================================
# 2️⃣ FUNZIONE: Algoritmo EBR
# ============================================================

def entropy_based_reduction(df, target_col, min_improvement=1e-5, max_features=None, force_continue=False):
    """
    Implementazione dell'Entropy-Based Reduction (EBR) per la selezione di feature.
    """
    # Rimuove eventuali spazi nei nomi delle colonne
    df.columns = df.columns.str.strip()

    # Esclude ID e target dal set di feature
    all_features = [f for f in df.columns if f not in [target_col, 'ID']]
    selected_features = []

    # Entropia iniziale del target H(Y)
    current_entropy = calculate_conditional_entropy(df, [], target_col)
    print(f"\n🧮 Entropia iniziale del target H(Y): {current_entropy:.4f}")

    step = 1
    while True:
        best_feature = None
        min_entropy = float('inf')

        # Trova la feature che riduce di più l'entropia
        for feature in all_features:
            if feature not in selected_features:
                temp_subset = selected_features + [feature]
                temp_entropy = calculate_conditional_entropy(df, temp_subset, target_col)

                if temp_entropy < min_entropy:
                    min_entropy = temp_entropy
                    best_feature = feature

        improvement = current_entropy - min_entropy

        # Condizione di stop
        if best_feature is not None and (improvement > min_improvement or force_continue):
            selected_features.append(best_feature)
            current_entropy = min_entropy
            print(f"\n🧩 Step {step}: Aggiunta feature '{best_feature}'")
            print(f"→ Entropia condizionale attuale: {current_entropy:.6f}")
            print(f"→ Riduzione ottenuta: {improvement:.6f}")
            print(f"→ Sottoinsieme corrente: {selected_features}")
            step += 1

            if max_features and len(selected_features) >= max_features:
                print("\n🔹 Raggiunto il numero massimo di feature consentite.")
                break
        else:
            print("\n⚙️ Nessuna riduzione significativa dell'entropia. Stop.")
            break

    print("\n✅ EBR completato.")
    print(f"Feature selezionate finali: {selected_features}")
    return selected_features


# ============================================================
# 3️⃣ PREPARAZIONE DEL DATASET
# ============================================================

# Rinomina del target (per evitare problemi con spazi)
df_robust.rename(columns={'DEFINITIVE DIAGNOSIS ': 'DIAGNOSIS'}, inplace=True)

print("\n🔧 Loading pre-discretized features...\n")

# Path to the pre-discretized features saved by QRA
discretized_features_path = os.path.join(lab_dir, 'QRA-discretized_features.csv')

# Load the discretized DataFrame
try:
    df_discretized = pd.read_csv(discretized_features_path)
    print(f"✅ Pre-discretized features loaded from '{discretized_features_path}'.")
except FileNotFoundError:
    print(f"❌ Error: Discretized features file not found at '{discretized_features_path}'. Please ensure QRA was run successfully.")
    raise

# Rename the target column in the loaded DataFrame from its original name to 'DIAGNOSIS'
# because the EBR function expects 'DIAGNOSIS' but the loaded CSV has 'DEFINITIVE DIAGNOSIS '
df_discretized.rename(columns={'DEFINITIVE DIAGNOSIS ': 'DIAGNOSIS'}, inplace=True)

print("✅ Dataset preparation complete.\n")


# ============================================================
# 4️⃣ ESECUZIONE DELL'ALGORITMO EBR
# ============================================================

print("--- Running Entropy-Based Reduction (EBR) ---")

ebr_reduct = entropy_based_reduction(
    df=df_discretized, # Use the discretized dataframe
    target_col='DIAGNOSIS',
    min_improvement=1e-5,
    force_continue=False  # set to True if you want to force more features even after H(Y|X)=0
)

# ============================================================
# 5️⃣ SALVATAGGIO RISULTATI
# ============================================================

reduct_df = pd.DataFrame({'EBR_selected_features': ebr_reduct})
reduct_df.to_csv(os.path.join(lab_dir, 'EBR-features.csv'), index=False)

dt_final_df = reduct_df.merge(icc_df[['feature', 'ICC(3,1)']], left_on='EBR_selected_features', right_on='feature', how='left')
dt_final_df.drop(columns=['feature'], inplace=True)

# Save to CSV
dt_final_df.to_csv(os.path.join(lab_dir, 'EBR-features_with_ICC.csv'), index=False)

print(f"\n📁 Results saved to: '{os.path.join(lab_dir, 'EBR-features.csv')}'")
print("-------------------------------------------------------------")


🔧 Loading pre-discretized features...

✅ Pre-discretized features loaded from '/content/drive/MyDrive/DSH - Lab/Lab7-8/QRA-discretized_features.csv'.
✅ Dataset preparation complete.

--- Running Entropy-Based Reduction (EBR) ---

🧮 Entropia iniziale del target H(Y): 0.9907

🧩 Step 1: Aggiunta feature 'original_shape_SurfaceVolumeRatio'
→ Entropia condizionale attuale: 0.738129
→ Riduzione ottenuta: 0.252607
→ Sottoinsieme corrente: ['original_shape_SurfaceVolumeRatio']

🧩 Step 2: Aggiunta feature 'original_shape_Maximum2DDiameterSlice'
→ Entropia condizionale attuale: 0.580286
→ Riduzione ottenuta: 0.157843
→ Sottoinsieme corrente: ['original_shape_SurfaceVolumeRatio', 'original_shape_Maximum2DDiameterSlice']

🧩 Step 3: Aggiunta feature 'original_glszm_ZonePercentage'
→ Entropia condizionale attuale: 0.279487
→ Riduzione ottenuta: 0.300798
→ Sottoinsieme corrente: ['original_shape_SurfaceVolumeRatio', 'original_shape_Maximum2DDiameterSlice', 'original_glszm_ZonePercentage']

🧩 Step 4:

# Wrapper methods - GA

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
import random

# --- Global Parameters for the Genetic Algorithm ---
POPULATION_SIZE = 50
NUM_GENERATIONS = 100
CROSSOVER_RATE = 0.8
MUTATION_RATE = 0.01
# Penalty for each feature included, to encourage smaller feature sets.
# This should be a small value, so it doesn't overpower the accuracy score.
FEATURE_PENALTY_ALPHA = 0.001

def fitness_function(individual, df, features, target_col):
    """
    Calculates the fitness of an individual (a feature subset).

    The fitness is calculated as the mean cross-validation accuracy of a
    Logistic Regression model, minus a small penalty for the number of selected features.

    Args:
        individual (list): A binary list representing the feature subset.
        df (pd.DataFrame): The full dataframe.
        features (list): The list of all possible feature names.
        target_col (str): The name of the target column.

    Returns:
        float: The fitness score for the individual.
    """
    selected_features = [features[i] for i, bit in enumerate(individual) if bit == 1]

    # If no features are selected, return a fitness of 0.
    if not selected_features:
        return 0.0

    X = df[selected_features]
    y = df[target_col]

    # Use a simple and fast classifier for fitness evaluation.
    classifier = LogisticRegression(max_iter=1000, random_state=42)

    # Use 5-fold cross-validation to get a robust accuracy estimate.
    try:
        accuracy = cross_val_score(classifier, X, y, cv=5, scoring='accuracy').mean()
    except ValueError:
        # This can happen if a feature has only one unique value in a CV split.
        return 0.0

    # Apply penalty for the number of features.
    penalty = FEATURE_PENALTY_ALPHA * len(selected_features)

    return accuracy - penalty

def tournament_selection(population, fitnesses):
    """Selects a parent using tournament selection."""
    tournament_size = 3
    # Select a few random individuals for the tournament.
    selection_ix = np.random.randint(len(population), size=tournament_size)
    # Find the one with the best fitness among them.
    best_ix = max(selection_ix, key=lambda i: fitnesses[i])
    return population[best_ix]

def crossover(p1, p2):
    """Performs single-point crossover."""
    c1, c2 = p1.copy(), p2.copy()
    if random.random() < CROSSOVER_RATE:
        # Select a random crossover point.
        pt = random.randint(1, len(p1) - 2)
        # Perform crossover.
        c1 = p1[:pt] + p2[pt:]
        c2 = p2[:pt] + p1[pt:]
    return c1, c2

def mutation(bitstring):
    """Performs bit-flip mutation."""
    for i in range(len(bitstring)):
        if random.random() < MUTATION_RATE:
            # Flip the bit.
            bitstring[i] = 1 - bitstring[i]

def genetic_algorithm_feature_selection(df, target_col, output_path):
    """
    Implements a Genetic Algorithm to find an optimal feature subset.

    Args:
        df (pd.DataFrame): The input dataframe.
        target_col (str): The name of the target class column.
        output_path (str): Path to save the final selected features CSV.
    """
    all_features = df.columns.drop([target_col, 'ID']).tolist()
    num_features = len(all_features)

    # --- 1. Initialize Population ---
    population = [[random.randint(0, 1) for _ in range(num_features)] for _ in range(POPULATION_SIZE)]

    best_individual, best_fitness = None, 0.0

    print("--- Starting Genetic Algorithm for Feature Selection ---")

    # --- 2. Main Evolution Loop ---
    for gen in range(NUM_GENERATIONS):
        # Calculate fitness for the entire population.
        fitnesses = [fitness_function(ind, df, all_features, target_col) for ind in population]

        # Check for a new best solution.
        for i in range(POPULATION_SIZE):
            if fitnesses[i] > best_fitness:
                best_individual = population[i]
                best_fitness = fitnesses[i]
                num_selected = sum(best_individual)
                print(f"> Generation {gen}: New Best! Fitness: {best_fitness:.4f}, Features: {num_selected}/{num_features}")

        # --- 3. Create Next Generation ---
        selected_parents = [tournament_selection(population, fitnesses) for _ in range(POPULATION_SIZE)]

        next_generation = []
        for i in range(0, POPULATION_SIZE, 2):
            p1, p2 = selected_parents[i], selected_parents[i+1]
            c1, c2 = crossover(p1, p2)
            mutation(c1)
            mutation(c2)
            next_generation.append(c1)
            next_generation.append(c2)

        population = next_generation

    print("\n--- Genetic Algorithm Finished ---")

    # --- 4. Decode and Save Best Solution ---
    final_selected_features = [all_features[i] for i, bit in enumerate(best_individual) if bit == 1]

    print(f"Final Best Fitness: {best_fitness:.4f}")
    print(f"Number of Selected Features: {len(final_selected_features)}")
    print(f"Selected Features: {final_selected_features}")

    # Save the results to a CSV file.
    features_df = pd.DataFrame(final_selected_features, columns=['selected_features'])
    try:
        features_df.to_csv(output_path, index=False)
        print(f"\nSuccessfully saved selected features to: {output_path}")

        root, ext = os.path.splitext(output_path)
        new_path = f"{root}_with_ICC.csv"

        ga_final_df = features_df.merge(icc_df[['feature', 'ICC(3,1)']], left_on='selected_features', right_on='feature', how='left')
        ga_final_df.drop(columns=['feature'], inplace=True)

        # Save to CSV
        ga_final_df.to_csv(new_path, index=False)

        print(f"\nRanked features with ICC saved to '{new_path}'")
    except Exception as e:
        print(f"\nError: Could not save the file. Please check the path '{output_path}'.")
        print(f"Details: {e}")

# Define the output path for the CSV file.
output_file_path = '/content/drive/MyDrive/DSH - Lab/Lab7-8/GA-features.csv'

# Run the Genetic Algorithm.
genetic_algorithm_feature_selection(df_robust, 'DIAGNOSIS', output_file_path)

--- Starting Genetic Algorithm for Feature Selection ---
> Generation 0: New Best! Fitness: 0.7003, Features: 46/86
> Generation 0: New Best! Fitness: 0.7114, Features: 44/86
> Generation 0: New Best! Fitness: 0.7365, Features: 38/86
> Generation 0: New Best! Fitness: 0.7575, Features: 36/86
> Generation 1: New Best! Fitness: 0.7626, Features: 40/86
> Generation 2: New Best! Fitness: 0.7736, Features: 39/86
> Generation 2: New Best! Fitness: 0.7756, Features: 37/86
> Generation 4: New Best! Fitness: 0.7776, Features: 35/86
> Generation 4: New Best! Fitness: 0.7786, Features: 34/86
> Generation 5: New Best! Fitness: 0.7787, Features: 33/86
> Generation 5: New Best! Fitness: 0.7821, Features: 40/86
> Generation 6: New Best! Fitness: 0.7831, Features: 39/86
> Generation 7: New Best! Fitness: 0.7871, Features: 35/86
> Generation 8: New Best! Fitness: 0.7901, Features: 32/86
> Generation 11: New Best! Fitness: 0.7911, Features: 31/86
> Generation 15: New Best! Fitness: 0.7921, Features: 30/

## GA results

--- Starting Genetic Algorithm for Feature Selection ---
> Generation 0: New Best! Fitness: 0.7259, Features: 49/91
> Generation 0: New Best! Fitness: 0.7316, Features: 42/91
> Generation 0: New Best! Fitness: 0.7430, Features: 41/91
> Generation 0: New Best! Fitness: 0.7431, Features: 50/91
> Generation 1: New Best! Fitness: 0.7475, Features: 46/91
> Generation 2: New Best! Fitness: 0.7485, Features: 45/91
> Generation 2: New Best! Fitness: 0.7515, Features: 42/91
> Generation 3: New Best! Fitness: 0.7515, Features: 42/91
> Generation 3: New Best! Fitness: 0.7545, Features: 39/91
> Generation 4: New Best! Fitness: 0.7580, Features: 45/91
> Generation 5: New Best! Fitness: 0.7590, Features: 44/91
> Generation 5: New Best! Fitness: 0.7826, Features: 39/91
> Generation 7: New Best! Fitness: 0.7836, Features: 38/91
> Generation 9: New Best! Fitness: 0.7846, Features: 37/91
> Generation 12: New Best! Fitness: 0.7856, Features: 36/91
> Generation 13: New Best! Fitness: 0.7866, Features: 35/91
> Generation 15: New Best! Fitness: 0.7876, Features: 34/91
> Generation 19: New Best! Fitness: 0.7886, Features: 33/91
> Generation 19: New Best! Fitness: 0.7896, Features: 32/91
> Generation 24: New Best! Fitness: 0.7906, Features: 31/91
> Generation 25: New Best! Fitness: 0.7916, Features: 30/91
> Generation 28: New Best! Fitness: 0.7926, Features: 29/91
> Generation 29: New Best! Fitness: 0.7936, Features: 28/91
> Generation 34: New Best! Fitness: 0.7946, Features: 27/91
> Generation 35: New Best! Fitness: 0.7956, Features: 26/91
> Generation 37: New Best! Fitness: 0.7966, Features: 25/91
> Generation 38: New Best! Fitness: 0.7976, Features: 24/91
> Generation 40: New Best! Fitness: 0.7986, Features: 23/91
> Generation 43: New Best! Fitness: 0.8006, Features: 21/91
> Generation 45: New Best! Fitness: 0.8016, Features: 20/91
> Generation 47: New Best! Fitness: 0.8026, Features: 19/91
> Generation 50: New Best! Fitness: 0.8036, Features: 18/91
> Generation 50: New Best! Fitness: 0.8056, Features: 16/91
> Generation 55: New Best! Fitness: 0.8061, Features: 16/91
> Generation 56: New Best! Fitness: 0.8142, Features: 17/91
> Generation 57: New Best! Fitness: 0.8152, Features: 16/91
> Generation 58: New Best! Fitness: 0.8162, Features: 15/91
> Generation 60: New Best! Fitness: 0.8257, Features: 15/91
> Generation 65: New Best! Fitness: 0.8277, Features: 13/91
> Generation 68: New Best! Fitness: 0.8298, Features: 20/91
> Generation 70: New Best! Fitness: 0.8352, Features: 15/91

--- Genetic Algorithm Finished ---
Final Best Fitness: 0.8352
Number of Selected Features: 15
Selected Features: ['original_glcm_Contrast', 'original_glcm_DifferenceEntropy', 'original_glrlm_GrayLevelVariance', 'original_glcm_ClusterTendency', 'original_glcm_ClusterProminence', 'original_firstorder_Uniformity', 'original_ngtdm_Strength', 'original_glszm_SmallAreaEmphasis', 'original_shape_SurfaceVolumeRatio', 'original_gldm_LargeDependenceHighGrayLevelEmphasis', 'original_glszm_ZoneEntropy', 'original_shape_LeastAxisLength', 'original_shape_Maximum3DDiameter', 'original_glcm_ClusterShade', 'original_firstorder_Skewness']


# Embedded method - Decision Tree

In [5]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier

def decision_tree_feature_selection(df, target_col, output_path):
    """
    Performs feature selection using the feature importances from a Decision Tree.

    This is an embedded method where the model itself determines the usefulness
    of each feature during the training process.

    Args:
        df (pd.DataFrame): The input dataframe.
        target_col (str): The name of the target class column.
        output_path (str): Path to save the final selected features CSV.
    """
    print("--- Starting Decision Tree Based Feature Selection ---")

    # 1. Prepare the data
    all_features = df.columns.drop([target_col, 'ID']).tolist()
    X = df[all_features]
    y = df[target_col]

    # 2. Train the Decision Tree Classifier
    # We use a simple Decision Tree to evaluate feature importance.
    # The random_state ensures reproducibility.
    dt_classifier = DecisionTreeClassifier(random_state=42)
    dt_classifier.fit(X, y)

    # 3. Get Feature Importances
    importances = dt_classifier.feature_importances_

    # Create a DataFrame to view feature importances
    feature_importance_df = pd.DataFrame({
        'feature': all_features,
        'importance': importances
    }).sort_values('importance', ascending=False)

    print("\nTop 10 Feature Importances:")
    print(feature_importance_df.head(10))

    # 4. Select features with importance > 0
    # A feature with an importance of 0 was not used by the tree at all.
    selected_features = feature_importance_df[feature_importance_df['importance'] > 0]['feature'].tolist()

    print(f"\nTotal features with importance > 0: {len(selected_features)}")
    print(f"Selected features: {selected_features}")

    # 5. Save the results to a CSV file
    features_df = pd.DataFrame(selected_features, columns=['selected_features'])
    try:
        features_df.to_csv(output_path, index=False)
        print(f"\nSuccessfully saved selected features to: {output_path}")

        root, ext = os.path.splitext(output_path)
        new_path = f"{root}_with_ICC.csv"

        dt_final_df = features_df.merge(icc_df[['feature', 'ICC(3,1)']], left_on='selected_features', right_on='feature', how='left')
        dt_final_df.drop(columns=['feature'], inplace=True)

        # Save to CSV
        dt_final_df.to_csv(new_path, index=False)

        print(f"\nRanked features with ICC saved to '{new_path}'")
    except Exception as e:
        print(f"\nError: Could not save the file. Please check the path '{output_path}'.")
        print(f"Details: {e}")

    print("\n--- Decision Tree Feature Selection Finished ---")
    return selected_features

# Define the output path for the CSV file.
output_file_path_dt = '/content/drive/MyDrive/DSH - Lab/Lab7-8/DT-features.csv'

# Run the Decision Tree feature selection.
dt_selected_features = decision_tree_feature_selection(df_robust, 'DIAGNOSIS', output_file_path_dt)

--- Starting Decision Tree Based Feature Selection ---


KeyError: "['DIAGNOSIS'] not found in axis"

--- Starting Decision Tree Based Feature Selection ---

Top 10 Feature Importances:
                                              feature  importance
54                  original_shape_SurfaceVolumeRatio    0.413580
70              original_shape_Maximum2DDiameterSlice    0.193175
45                                 original_glcm_Imc1    0.073418
72       original_glszm_SmallAreaLowGrayLevelEmphasis    0.064506
48                            original_ngtdm_Strength    0.061434
84                   original_shape_Maximum3DDiameter    0.061221
64                    original_firstorder_TotalEnergy    0.038226
43  original_gldm_SmallDependenceLowGrayLevelEmphasis    0.037101
62                     original_shape_MajorAxisLength    0.028669
71              original_glrlm_RunLengthNonUniformity    0.028669

Total features with importance > 0: 10
Selected features: ['original_shape_SurfaceVolumeRatio', 'original_shape_Maximum2DDiameterSlice', 'original_glcm_Imc1', 'original_glszm_SmallAreaLowGrayLevelEmphasis', 'original_ngtdm_Strength', 'original_shape_Maximum3DDiameter', 'original_firstorder_TotalEnergy', 'original_gldm_SmallDependenceLowGrayLevelEmphasis', 'original_shape_MajorAxisLength', 'original_glrlm_RunLengthNonUniformity']